In [ ]:
import os
import json
import time
import copy
import logging
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import skew

# ----------------------- Dataset & utilities -----------------------

def setup_logging(log_file='grid_search.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger()
    root_logger.handlers = []
    root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w')
    file_handler.setFormatter(log_formatter)
    root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(log_formatter)
    root_logger.addHandler(console_handler)


class RegressionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float().view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [2]:
def get_housing_data_loaders(batch_size=128, test_size=0.2):
    """
    使用固定 Kaggle 路径读取数据，返回 (X_train, y_train), (X_val, y_val), input_dim
    """
    logging.info("Preparing Ames Housing dataset...")

    # >>> 固定路径，唯一改动处 <<<
    train_path = '/kaggle/input/house-prices-advanced-regression-techniques/train.csv'
    test_path  = '/kaggle/input/house-prices-advanced-regression-techniques/test.csv'

    train_df = pd.read_csv(train_path)
    test_df  = pd.read_csv(test_path)
    # ----------------------------------

    full_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

    # 缺失值处理
    fill_none = ["Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
                 "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond", "PoolQC",
                 "Fence", "MiscFeature", "MasVnrType"]
    for col in fill_none:
        if col in full_df.columns:
            full_df[col] = full_df[col].fillna("None")

    fill_zero = ["BsmtFullBath", "BsmtHalfBath", "BsmtUnfSF", "GarageArea", "GarageCars", "MasVnrArea",
                 "PoolArea", "MiscVal", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch",
                 "TotRmsAbvGrd", "BedroomAbvGr", "Fireplaces", "HalfBath", "FullBath", "KitchenAbvGr",
                 "LotFrontage", "WoodDeckSF"]
    for col in fill_zero:
        if col in full_df.columns:
            full_df[col] = full_df[col].fillna(0)

    num_cols = full_df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if full_df[col].isnull().sum() > 0:
            full_df[col] = full_df[col].fillna(full_df[col].median())

    # 丢弃低价值列
    cols_to_drop = ['Id', 'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                    'LotFrontage', 'GarageYrBlt', 'MoSold', 'YrSold']
    full_df = full_df.drop(columns=cols_to_drop, errors='ignore')

    # 特征工程
    full_df["TotalBath"] = (full_df["BsmtFullBath"] + 0.5 * full_df["BsmtHalfBath"] +
                            full_df["FullBath"] + 0.5 * full_df["HalfBath"])
    full_df["AllSF"]   = full_df["GrLivArea"] + full_df["TotalBsmtSF"]
    full_df["AllFlrsSF"] = full_df["1stFlrSF"] + full_df["2ndFlrSF"]
    full_df["AllPorchSF"] = (full_df["OpenPorchSF"] + full_df["EnclosedPorch"] +
                             full_df["3SsnPorch"] + full_df["ScreenPorch"])

    # 独热编码
    categorical_cols = full_df.select_dtypes(include=['object']).columns
    full_df = pd.get_dummies(full_df, columns=categorical_cols)

    # 处理偏度
    numeric_feats = full_df.select_dtypes(include=[np.number]).columns
    skewed_feats  = full_df[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
    skewness = skewed_feats[abs(skewed_feats) > 0.5]
    for feat in skewness.index:
        full_df[feat] = np.log1p(full_df[feat])

    # 重新切回 train / test
    train_df = full_df.iloc[:len(train_df), :].copy()
    test_df  = full_df.iloc[len(train_df):, :].drop(columns=['SalePrice'], errors='ignore')

    X = train_df.drop(columns=['SalePrice'])
    # >>> 关键修正：把 Series 转成 ndarray <<<
    y = np.log1p(train_df['SalePrice']).values

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    X_train, X_val, y_train, y_val = train_test_split(
        X.astype(np.float32), y.astype(np.float32),
        test_size=test_size, random_state=42)

    logging.info(f"Prepared. Train: {len(X_train)}, Val: {len(X_val)}, Features: {X_train.shape[1]}")
    return (X_train, y_train), (X_val, y_val), X_train.shape[1]

In [ ]:
# ----------------------- Model -----------------------
class PolynomialTerm(nn.Module):
    def __init__(self, input_dim, output_dim, degree):
        super().__init__()
        self.degree = degree
        self.projection = nn.Linear(input_dim, output_dim)
        self.coeffs = nn.Parameter(torch.randn(output_dim, degree + 1) * 0.1)

    def forward(self, x):
        z = torch.tanh(self.projection(x))
        powers = [torch.ones_like(z)]
        for d in range(1, self.degree + 1):
            powers.append(powers[-1] * z)
        z_powered = torch.stack(powers, dim=-1)          # [B, output_dim, degree+1]
        output = torch.sum(z_powered * self.coeffs, dim=2)  # [B, output_dim]
        return output


class CFNet(nn.Module):
    def __init__(self, input_dim, output_dim, depth, poly_degree):
        super().__init__()
        self.output_dim = output_dim
        self.terms = nn.ModuleList([
            PolynomialTerm(input_dim, output_dim, poly_degree) for _ in range(depth)
        ])
        self.raw_betas = nn.ParameterList([
            nn.Parameter(torch.full((output_dim,), 0.1)) for _ in range(depth - 1)
        ])

    def forward(self, x):
        if not self.terms:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        output = F.softplus(self.terms[-1](x)) + 1.0
        for i in range(len(self.terms) - 2, -1, -1):
            beta = F.softplus(self.raw_betas[i])
            output = self.terms[i](x) + beta / output
        return output


class EnsembleResCoFrNet(nn.Module):
    def __init__(self, input_dim, output_dim, shallow_depth, poly_degree, learning_rate):
        super().__init__()
        self.models = nn.ModuleList()
        self.candidate_model = None
        self.lr = learning_rate
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.shallow_depth = shallow_depth
        self.poly_degree = poly_degree

    def forward(self, x):
        if not self.models and self.candidate_model is None:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        total = torch.zeros(x.shape[0], self.output_dim, device=x.device)
        for m in self.models:
            total += self.lr * m(x)
        if self.candidate_model is not None:
            total += self.lr * self.candidate_model(x)
        return total

    def create_candidate(self):
        self.candidate_model = CFNet(
            self.input_dim, self.output_dim,
            self.shallow_depth, self.poly_degree)
        return self.candidate_model

    def promote_candidate(self):
        if self.candidate_model is not None:
            self.models.append(self.candidate_model)
            self.candidate_model = None

    def discard_candidate(self):
        self.candidate_model = None

    def freeze_for_candidate_training(self):
        for m in self.models:
            for p in m.parameters():
                p.requires_grad = False
        if self.candidate_model is not None:
            for p in self.candidate_model.parameters():
                p.requires_grad = True


# ----------------------- Regressor -----------------------
class CoFrNetRegressor:
    def __init__(self, hparams):
        self.hparams = hparams
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.ensemble = EnsembleResCoFrNet(
            hparams['input_dim'], hparams['output_dim'],
            hparams['shallow_depth_per_cofrnet'], hparams['polynomial_degree'],
            hparams['boosting_learning_rate']).to(self.device)

        self.best_state = None
        self.best_rmse  = float('inf')
        self.best_size  = 0

    def train(self, train_data, val_data, test_data, log_filepath):
        X_tr, y_tr = train_data
        X_va, y_va = val_data
        X_te, y_te = test_data

        train_set = RegressionDataset(X_tr, y_tr)
        val_set   = RegressionDataset(X_va, y_va)
        test_set  = RegressionDataset(X_te, y_te)

        tr_loader = DataLoader(train_set, batch_size=self.hparams['batch_size'], shuffle=True)
        va_loader = DataLoader(val_set,   batch_size=self.hparams['batch_size'], shuffle=False)
        te_loader = DataLoader(test_set,  batch_size=self.hparams['batch_size'], shuffle=False)

        criterion = nn.MSELoss()
        patience  = self.hparams.get('early_stopping_patience', 10)

        logging.info("Starting selective-add ensemble training...")

        for attempt in range(self.hparams['num_models_max']):
            logging.info(f"Attempt {attempt+1}/{self.hparams['num_models_max']}  "
                         f"| current size: {len(self.ensemble.models)}")
            cand = self.ensemble.create_candidate().to(self.device)
            self.ensemble.freeze_for_candidate_training()

            opt = optim.Adam(cand.parameters(),
                             lr=self.hparams['learning_rate_adam'],
                             weight_decay=self.hparams['weight_decay'])
            sched = optim.lr_scheduler.CosineAnnealingLR(
                opt, T_max=self.hparams['epochs_per_model'])

            epochs_no_imp = 0
            best_va_rmse  = float('inf')
            best_cand_sd  = None

            for epoch in range(self.hparams['epochs_per_model']):
                self.ensemble.train()
                running = 0.0
                for xb, yb in tr_loader:
                    xb, yb = xb.to(self.device), yb.to(self.device)
                    opt.zero_grad()
                    out = self.ensemble(xb)
                    loss = criterion(out, yb)
                    loss.backward()
                    opt.step()
                    running += loss.item()
                sched.step()

                va_rmse = self.evaluate(va_loader)
                if va_rmse < best_va_rmse:
                    best_va_rmse = va_rmse
                    epochs_no_imp = 0
                    best_cand_sd  = copy.deepcopy(cand.state_dict())
                else:
                    epochs_no_imp += 1
                if epochs_no_imp >= patience:
                    if best_cand_sd:
                        cand.load_state_dict(best_cand_sd)
                    break

            te_rmse_cand = self.evaluate(te_loader)
            accepted = te_rmse_cand < self.best_rmse

            if accepted:
                self.best_rmse = te_rmse_cand
                self.ensemble.promote_candidate()
                self.best_size = len(self.ensemble.models)
                self.best_state = copy.deepcopy(self.ensemble.state_dict())
                logging.info(f"Accepted | new best RMSE: {self.best_rmse:.6f}  "
                             f"| size: {self.best_size}")
            else:
                self.ensemble.discard_candidate()
                logging.info(f"Rejected | RMSE with cand: {te_rmse_cand:.6f}  "
                             f"| best so far: {self.best_rmse:.6f}")

        if self.best_state:
            self.ensemble.load_state_dict(self.best_state)
        return self.best_rmse

    @torch.no_grad()
    def evaluate(self, loader):
        self.ensemble.eval()
        total_mse = 0.0
        crit = nn.MSELoss()
        for xb, yb in loader:
            xb, yb = xb.to(self.device), yb.to(self.device)
            out = self.ensemble(xb)
            total_mse += crit(out, yb).item() * xb.size(0)
        return np.sqrt(total_mse / len(loader.dataset))

    @torch.no_grad()
    def predict(self, X):
        self.ensemble.eval()
        X_t = torch.from_numpy(X).float().to(self.device)
        return self.ensemble(X_t).cpu().numpy()


# ----------------------- Grid search & plotting -----------------------
def grid_search(depths, degrees, base_hparams, results_path='grid_results.json'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    (X_tr, y_tr), (X_va, y_va), inp_dim = get_housing_data_loaders(
        batch_size=base_hparams['batch_size'])
    base_hparams['input_dim'] = inp_dim

    results = np.zeros((len(degrees), len(depths)), dtype=float)
    details = {}

    for i, deg in enumerate(degrees):
        for j, dep in enumerate(depths):
            logging.info(f"Grid point >>> degree={deg}  depth={dep}")
            hp = copy.deepcopy(base_hparams)
            hp['polynomial_degree']        = deg
            hp['shallow_depth_per_cofrnet'] = dep

            reg = CoFrNetRegressor(hp)
            t0  = time.time()
            rmse = reg.train((X_tr, y_tr), (X_va, y_va), (X_va, y_va),
                             log_filepath=f'log_d{deg}_dp{dep}.txt')
            elapsed = time.time() - t0

            results[i, j] = rmse
            details[f"{deg}_{dep}"] = {
                "rmse": float(rmse),
                "time": elapsed,
                "ensemble_size": reg.best_size}
            logging.info(f"<<< degree={deg} depth={dep} | RMSE={rmse:.6f}  "
                         f"time={elapsed:.1f}s  size={reg.best_size}")

            del reg
            torch.cuda.empty_cache()

    with open(results_path, 'w') as f:
        json.dump({'depths': depths,
                   'degrees': degrees,
                   'results_matrix': results.tolist(),
                   'details': details}, f, indent=2)
    return results, degrees, depths


def plot_heatmap(results, degrees, depths, save_path='heatmap_depth_vs_degree.png'):
    plt.figure(figsize=(max(6, len(depths)*0.6), max(4, len(degrees)*0.6)))
    sns.heatmap(results,
                xticklabels=depths, yticklabels=degrees,
                annot=True, fmt='.4f', cmap='viridis')
    plt.xlabel('Depth (shallow depth per CFNet)')
    plt.ylabel('Polynomial Degree')
    plt.title('CFNet-Boost Grid Search RMSE (lower is better)')
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    logging.info(f'Saved heatmap -> {save_path}')


# ----------------------- Main -----------------------
if __name__ == '__main__':
    setup_logging('grid_search.log')

    BASE_HPARAMS = {
        'task_name': 'AmesHousing',
        'num_models_max': 15,
        'epochs_per_model': 200,
        'shallow_depth_per_cofrnet': 2,
        'polynomial_degree': 2,
        'boosting_learning_rate': 0.1,
        'learning_rate_adam': 1e-3,
        'weight_decay': 1e-5,
        'batch_size': 128,
        'early_stopping_patience': 10,
        'input_dim': None,
        'output_dim': 1
    }

    depths = [1, 2, 3, 4,5,6,7,8]
    degrees = [1, 2, 3, 4,5,6,7,8]

    results, degrees, depths = grid_search(depths, degrees, BASE_HPARAMS)
    plot_heatmap(results, degrees, depths)

    print('Grid search complete. Results saved to grid_results.json and heatmap file.')

2025-10-18 10:35:01,082 [INFO ]  Preparing Ames Housing dataset...
2025-10-18 10:35:01,383 [INFO ]  Prepared. Train: 1168, Val: 292, Features: 279
2025-10-18 10:35:01,385 [INFO ]  Grid point >>> degree=1  depth=1
2025-10-18 10:35:01,410 [INFO ]  Starting selective-add ensemble training...
2025-10-18 10:35:01,412 [INFO ]  Attempt 1/15  | current size: 0
2025-10-18 10:35:11,277 [INFO ]  Accepted | new best RMSE: 2.235846  | size: 1
2025-10-18 10:35:11,278 [INFO ]  Attempt 2/15  | current size: 1
2025-10-18 10:35:17,612 [INFO ]  Accepted | new best RMSE: 1.900033  | size: 2
2025-10-18 10:35:17,614 [INFO ]  Attempt 3/15  | current size: 2
2025-10-18 10:35:24,407 [INFO ]  Accepted | new best RMSE: 1.584402  | size: 3
2025-10-18 10:35:24,408 [INFO ]  Attempt 4/15  | current size: 3
2025-10-18 10:35:31,576 [INFO ]  Accepted | new best RMSE: 1.258474  | size: 4
2025-10-18 10:35:31,577 [INFO ]  Attempt 5/15  | current size: 4
2025-10-18 10:35:39,391 [INFO ]  Accepted | new best RMSE: 0.939776  